In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('outage.csv', skiprows=5)
df = df.drop('variables', axis=1)
df = df.iloc[1:]

In [4]:
df.isna().sum()[df.isna().sum() > 0]

,0
MONTH,9
CLIMATE.REGION,6
ANOMALY.LEVEL,9
CLIMATE.CATEGORY,9
OUTAGE.START.DATE,9
OUTAGE.START.TIME,9
OUTAGE.RESTORATION.DATE,58
OUTAGE.RESTORATION.TIME,58
CAUSE.CATEGORY.DETAIL,471
HURRICANE.NAMES,1462


CUSTOMERS.AFFECTED could be MNAR. This data could fail to be reported when the magnitude of the impact is very large or very small.

In [5]:
col = "DEMAND.LOSS.MW"
df = df.copy()

# Missingness indicator: True if DEMAND.LOSS.MW is missing
df["miss_demand"] = df[col].isna()
df["miss_demand"].value_counts(dropna=False)

,count
miss_demand,
False,829
True,705


In [6]:
df["OUTAGE.DURATION"].dtype
df["OUTAGE.DURATION"].head(10)
df["OUTAGE.DURATION"].astype(str).str.slice(0, 40).value_counts().head(10)

,count
OUTAGE.DURATION,
1,97
0,78
nan,58
2880,15
300,14
60,14
1440,13
15,9
4320,9


In [7]:
df["OUTAGE.DURATION_NUM"] = pd.to_numeric(df["OUTAGE.DURATION"], errors="coerce")
df["OUTAGE.DURATION_NUM"].describe()

,OUTAGE.DURATION_NUM
count,1476.000000
mean,2625.398374
std,5942.483307
min,0.000000
25%,102.250000
50%,701.000000
75%,2880.000000
max,108653.000000


In [8]:
x = "OUTAGE.DURATION_NUM"

tmp = df.loc[df[x].notna(), ["miss_demand", x]].copy()

obs = (
    tmp.loc[tmp["miss_demand"], x].mean()
    - tmp.loc[~tmp["miss_demand"], x].mean()
)

obs


def perm_test_diff_means(tmp, group_col, value_col, n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)
    g = tmp[group_col].to_numpy()
    v = tmp[value_col].to_numpy()

    obs = v[g].mean() - v[~g].mean()

    perm_stats = np.empty(n_perm)
    for i in range(n_perm):
        g_perm = rng.permutation(g)
        perm_stats[i] = v[g_perm].mean() - v[~g_perm].mean()

    # two-sided p-value
    p = (np.abs(perm_stats) >= np.abs(obs)).mean()
    return obs, perm_stats, p

obs_stat, perm_stats, pval = perm_test_diff_means(tmp, "miss_demand", x, n_perm=5000, seed=42)
obs_stat, pval

(np.float64(128.76779051172707), np.float64(0.6824))

There is no statistical evidence that missingness of demand.loss.mw depends on outage duration. (p=0.6824 >= 0.05)

In [9]:
import plotly.express as px
import numpy as np

plot_df = tmp.copy()
plot_df["group"] = plot_df["miss_demand"].map({
    True: "DEMAND.LOSS.MW missing",
    False: "DEMAND.LOSS.MW present"
})

plot_df["log_duration"] = np.log1p(plot_df[x])

fig = px.histogram(
    plot_df,
    x="log_duration",
    color="group",
    nbins=60,
    barmode="overlay",
    histnorm="probability density",
    title="Log Outage Duration by DEMAND.LOSS.MW Missingness"
)

fig.update_layout(
    xaxis_title="log(1 + OUTAGE.DURATION)",
    yaxis_title="Density"
)

fig.show()

In [10]:
from scipy.stats import chi2_contingency

cat = "CAUSE.CATEGORY"

tmpc = df.loc[df[cat].notna(), ["miss_demand", cat]].copy()

In [11]:
ct = pd.crosstab(tmpc[cat], tmpc["miss_demand"])
chi2_obs = chi2_contingency(ct, correction=False)[0]
chi2_obs

np.float64(101.3661710576918)

In [12]:
import numpy as np

def perm_test_chi2(tmpc, group_col, cat_col, n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)

    g = tmpc[group_col].to_numpy()
    c = tmpc[cat_col].to_numpy()

    def chi2_stat(gvec):
        ct = pd.crosstab(c, gvec)
        return chi2_contingency(ct, correction=False)[0]

    obs = chi2_stat(g)
    perm_stats = np.array([chi2_stat(rng.permutation(g)) for _ in range(n_perm)])
    p = (perm_stats >= obs).mean()
    return obs, perm_stats, p

chi2_obs, perm_stats_cat, pval_cat = perm_test_chi2(
    tmpc, "miss_demand", cat, n_perm=5000, seed=42
)

chi2_obs, pval_cat

(np.float64(101.3661710576918), np.float64(0.0))

The missingness of DEMAND.LOSS.MW depends on CAUSE.CATEGORY.<br>